# Проект: Прогнозирование стоимости недвижимости

## Этап 1. Исследование данных (EDA)

### 1.1 Знакомство с данными

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [ ]:
import os

data_path = 'real_estate_data.csv' if os.path.exists('real_estate_data.csv') else 'данные для работы/real_estate_data.csv'
df = pd.read_csv(data_path, low_memory=False)
df.shape

В датасете 403 487 строк и 17 столбцов.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(1, random_state=1)

Датасет содержит объявления о продаже и аренде недвижимости с турецкого сервиса Zingat. Это площадка-посредник между покупателями/арендаторами и продавцами/арендодателями недвижимости.

Описание признаков:
- id — номер объявления;
- type — тип недвижимости (в датасете только одно значение — Konut, т.е. жильё);
- sub_type — подтип жилья (Daire — квартира, Villa — вилла и т.д.);
- start_date / end_date — даты публикации и снятия объявления;
- listing_type — тип объявления: 1 — продажа, 2 — аренда, 3 — прочее;
- tom — сколько дней объявление находилось на рынке (time on market);
- building_age — возраст здания;
- total_floor_count — этажность дома;
- floor_no — этаж, на котором расположена квартира;
- room_count — количество комнат (например, 2+1 — 2 комнаты + гостиная);
- size — площадь в м²;
- address — адрес в формате город/район/квартал;
- furnished — меблировка;
- heating_type — тип отопления;
- price — цена (целевая переменная);
- price_currency` — валюта цены.

In [ ]:
df.info()

Всего 3 числовых столбца (float64), 3 столбца int64 и 11 столбцов типа object. По df.info() сразу видно, что часть столбцов (end_date, building_age, size, furnished и др.) содержат пропуски.

In [ ]:
# Количество уникальных значений в каждом признаке
df.nunique()

По уникальным значениям видно:
- `type` содержит всего 1 уникальное значение (`Konut`), то есть признак константный и не несет полезной информации;
- `furnished` содержит 0 уникальных значений (он полностью пустой);
- `listing_type` имеет 3 значения (продажа, аренда, посуточная аренда);
- `id` и `address` имеют сотни тысяч уникальных значений — `id` нужно удалить, а адрес разделить на составляющие.

### 1.2 Описательная статистика

In [ ]:
df.describe()

По числовым столбцам:
- listing_type — медиана 1, то есть среди объявлений преобладает продажа (1), аренда (2) встречается реже;
- tom — объявление в среднем находится на рынке 57 дней, максимум ограничен 180 днями;
- size — здесь явно есть выбросы: медиана 110 м², а максимум почти 950 000 м², так быть не может;
- furnished — count = 0, то есть столбец полностью пустой, толку от него не будет;
- price — та же история, что и с площадью: медиана ≈ 199 000, а максимум — 2 млрд, огромные выбросы.

In [ ]:
df.describe(include='object')

По категориальным столбцам:
- type — всего 1 уникальное значение (Konut), для модели это бесполезный признак;
- sub_type — 12 категорий, чаще всего встречается Daire (квартира);
- room_count — больше всего вариантов 3+1;
- address — очень много уникальных адресов (это, по сути, город/район/квартал одной строкой), напрямую в модель такой признак не пойдёт, нужно будет разбивать.

In [ ]:
print('Асимметрия цены:', df['price'].skew())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(df.loc[df['price'] > 0, 'price'], bins=50, color='steelblue')
plt.title('Распределение цены (обычная шкала)')
plt.show()

Асимметрия цены огромная, и на гистограмме в обычной шкале это отлично видно: почти все столбики прижаты к нулю, а редкие огромные выбросы растягивают ось так, что основное распределение не разглядеть. Перед моделированием цену, скорее всего, нужно будет логарифмировать — в логарифмической шкале распределение выглядит гораздо адекватнее.

In [ ]:
# Подробная описательная статистика целевой переменной (цена)
price_pos = df.loc[df['price'] > 0, 'price']
q25 = price_pos.quantile(0.25)
q75 = price_pos.quantile(0.75)
iqr = q75 - q25

price_stats_df = pd.DataFrame({
    'Метрика': ['Минимум (min)', 'Максимум (max)', 'Среднее (mean)', 'Медиана (median)', 
                'Стандартное отклонение (std)', 'Межквартильный размах (IQR)', 'Асимметрия (skewness)'],
    'Значение': [f"{price_pos.min():,.2f}", f"{price_pos.max():,.2f}", f"{price_pos.mean():,.2f}", 
                 f"{price_pos.median():,.2f}", f"{price_pos.std():,.2f}", f"{iqr:,.2f}", f"{price_pos.skew():.2f}"]
})
display(price_stats_df)

In [ ]:
# Построение Boxplot цены и проверка эффекта от log1p(price)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot цены (исходный масштаб до 5 млн для наглядности)
sns.boxplot(x=df.loc[(df['price'] > 0) & (df['price'] <= 5_000_000), 'price'], ax=axes[0], color='lightcoral')
axes[0].set_title('Boxplot цены (до 5 млн TRY)')
axes[0].set_xlabel('Цена (TRY)')

# Распределение после применения log1p
log_price = np.log1p(price_pos)
sns.histplot(log_price, bins=50, kde=True, ax=axes[1], color='mediumseagreen')
axes[1].set_title(f'Гистограмма log1p(price) (асимметрия = {log_price.skew():.2f})')
axes[1].set_xlabel('log1p(Цена)')

plt.tight_layout()
plt.show()

**Краткий вывод по целевой переменной (цена):**
- В исходной шкале средняя цена (~470 тыс. TRY) намного больше медианы (~199 тыс. TRY), а асимметрия огромная (56.7).
- На ящике с усами (boxplot) виден длиннющий шлейф выбросов справа вплоть до нескольких миллиардов.
- После применения функции `log1p(price)` асимметрия падает до 0.44, а само распределение становится близким к нормальному колоколообразному виду. Поэтому для моделей регрессии логарифмирование целевой переменной строго необходимо.

### 1.3 Визуальный анализ

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(df['size'].dropna(), bins=50, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Площадь (size)')

sns.histplot(df['tom'], bins=50, ax=axes[0, 1], color='teal')
axes[0, 1].set_title('Дней на рынке (tom)')

sns.histplot(df.loc[df['price'] > 0, 'price'], bins=50, log_scale=True, ax=axes[1, 0], color='teal')
axes[1, 0].set_title('Цена (лог. шкала)')

df['listing_type'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='teal')
axes[1, 1].set_title('Тип объявления')

plt.tight_layout()
plt.show()

- size — гистограмма сжата в одну точку слева из-за редких аномально больших значений;
- tom — резкие пики на 30, 60, 90, 120, 180 днях, похоже, что площадка сама снимает объявления по этим срокам;
- price в логарифмической шкале — видно два «горба»: слева это аренда (цены поменьше), справа — продажа;
- listing_type — продажа (1) встречается гораздо чаще аренды (2), тип 3 почти не встречается.

In [ ]:
import re

def parse_rooms(value):
    if pd.isna(value):
        return np.nan
    nums = re.findall(r'\d+', str(value))
    if len(nums) == 2:
        return int(nums[0]) + int(nums[1])
    return np.nan

df['rooms_total'] = df['room_count'].apply(parse_rooms)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(df['rooms_total'].dropna(), bins=range(1, 15), ax=axes[0], color='darkorange')
axes[0].set_title('Количество комнат (всего)')

age_order = df['building_age'].value_counts().index
sns.countplot(data=df, y='building_age', order=age_order, ax=axes[1], color='darkorange')
axes[1].set_title('Возраст здания (building_age)')

plt.tight_layout()
plt.show()

Чаще всего встречаются квартиры с 3-4 комнатами всего (то есть 2+1 и 3+1). По возрасту здания больше всего новостроек ('0' — только что построенные), дальше идут дома возрастом 6-10 лет.

In [ ]:
sample = df[df['size'].between(10, 500) & df['price'].between(1000, 3_000_000)].sample(3000, random_state=1)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample, x='size', y='price', hue='listing_type', alpha=0.4, palette='Set1')
plt.title('Площадь vs Цена')
plt.show()

Чем больше площадь, тем выше цена — зависимость видна, хотя разброс большой. Также хорошо видно, что точки аренды (listing_type=2) лежат заметно ниже точек продажи (listing_type=1) — это две разные ценовые категории.

In [ ]:
sample2 = df[df['price'].between(1000, 3_000_000) & df['rooms_total'].between(1, 10)].sample(3000, random_state=1)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample2, x='rooms_total', y='price', alpha=0.3, color='darkorange')
plt.title('Количество комнат vs Цена')
plt.show()

Цена тоже растёт вместе с количеством комнат, но тут разброс ещё больше, чем у площади — скорее всего потому что количество комнат само по себе грубее площади (например, 3+1 может быть и 70 м², и 130 м²).

In [ ]:
df['city'] = df['address'].str.split('/').str[0]
top_cities = df['city'].value_counts().head(6).index

plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df[df['city'].isin(top_cities) & df['price'].between(1000, 3_000_000)],
    x='city', y='price'
)
plt.xticks(rotation=20)
plt.title('Цена по городам (топ-6 по числу объявлений)')
plt.show()

Из адреса выделила город. В топ-6 городов по числу объявлений median-цена сильно различается: например, в Айдыне и Измире цены выше, чем в Стамбуле и Анкаре, хотя объявлений в Стамбуле больше всего. Значит, город/район — важный признак.

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=df[df['size'].between(1, 1000)]['size'], color='lightgreen')
plt.title('Boxplot площади (проверка выбросов)')
plt.show()

Даже если сразу ограничить площадь до 1000 м², boxplot всё равно показывает огромное количество точек-выбросов справа. Это ещё раз подтверждает, что на этапе предобработки площадь нужно будет чистить от аномалий.

In [ ]:
num_cols = ['listing_type', 'tom', 'size', 'rooms_total', 'price']
filtered = df[df['size'].between(10, 500) & df['price'].between(1000, 3_000_000)]

plt.figure(figsize=(6, 5))
sns.heatmap(filtered[num_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Корреляция числовых признаков (после отсечения выбросов)')
plt.show()

Сначала я считала корреляцию на «сырых» данных, но из-за огромных выбросов в price и size она получалась почти нулевой и бессмысленной. После отсечения явных аномалий (площадь 10–500 м², цена 1000–3 000 000) картина стала понятнее:
- listing_type и price — заметная отрицательная связь (-0.44): у аренды (2) цена меньше, чем у продажи (1);
- size и price — положительная связь (+0.39): больше площадь — выше цена;
- rooms_total и price — тоже положительная связь (+0.36), похожая на площадь, что логично — это связанные признаки;
- tom почти не связан с ценой.

In [ ]:
mean_price = filtered.groupby('sub_type')['price'].mean().sort_values()

plt.figure(figsize=(8, 5))
mean_price.plot(kind='barh', color='teal')
plt.title('Средняя цена по типу жилья (sub_type)')
plt.xlabel('Средняя цена')
plt.show()

Самые дешёвые в среднем — сборные дома (Prefabrik Ev), самые дорогие — особняки/усадьбы (Köşk/Konak/Yalı) и виллы. Обычная квартира (Daire, самая частая категория) по цене где-то в середине списка.

In [ ]:
# Круговая диаграмма (Pie Chart) распределения типов недвижимости
sub_type_counts = df['sub_type'].value_counts()
threshold = 0.015 * len(df)
main_types = sub_type_counts[sub_type_counts >= threshold]
other_types = pd.Series({'Прочее': sub_type_counts[sub_type_counts < threshold].sum()})
pie_data = pd.concat([main_types, other_types])

plt.figure(figsize=(7, 7))
plt.pie(pie_data, labels=pie_data.index, autopct='%1.1f%%', startangle=140, 
        colors=sns.color_palette('pastel'), explode=[0.05] + [0]*(len(pie_data)-1))
plt.title('Распределение типов недвижимости (Pie Chart)')
plt.show()

**Краткий вывод по типам недвижимости (Pie Chart):**
Почти три четверти всех объявлений (73%) — это обычные квартиры (`Daire`). На втором месте идут виллы (`Villa`, 14.8%), на третьем — дачные дома (`Yazlık`, 4.4%) и частные дома (`Müstakil Ev`, 4.0%). Редкие категории вроде лофтов и ферм составляют менее 4% суммарно.

In [ ]:
# Средняя и медианная цена по типам отопления
sample_clean_heat = df[df['price'].between(1000, 3_000_000)]
heating_stats = sample_clean_heat.groupby('heating_type')['price'].agg(['mean', 'median']).sort_values('median', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(heating_stats))
width = 0.35

ax.bar(x - width/2, heating_stats['mean'], width, label='Средняя цена', color='steelblue')
ax.bar(x + width/2, heating_stats['median'], width, label='Медианная цена', color='darkorange')

ax.set_title('Средняя и медианная цена по типам отопления (heating_type)')
ax.set_ylabel('Цена (TRY)')
ax.set_xticks(x)
ax.set_xticklabels(heating_stats.index, rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

**Краткий вывод по типам отопления:**
Тип отопления четко дифференцирует жилье по цене. Самые дорогие объекты оборудованы теплыми полами (`Yerden Isıtma`) и этажными котлами (`Kat Kaloriferi`), а самые дешевые — угольными печами (`Soba Kömür`) или не имеют отопления вовсе (`Yok`). При этом среднее всегда заметно выше медианы.

### 1.4 Анализ пропусков и аномалий

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'пропусков': missing, 'процент, %': missing_pct}).sort_values('процент, %', ascending=False)

Больше всего пропусков в furnished — 100%, столбец совсем без данных, на следующем этапе его логично удалить. Дальше по количеству пропусков: end_date (34%), size (36%), floor_no (9%), total_floor_count и heating_type (7%), building_age (7%). У price/price_currency пропусков мало (715 строк).

In [ ]:
# Матрица пропусков (визуализация пропущенных значений на выборке из 1000 строк)
plt.figure(figsize=(11, 5))
sns.heatmap(df.sample(1000, random_state=42).isna(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Матрица пропущенных значений (желтые полосы — пропуски)')
plt.show()

**Краткий вывод по матрице пропусков:**
Наглядно видно, что признак `furnished` состоит из одних пропусков (желтая сплошная колонка). Также системные пропуски наблюдаются в `end_date` (активные объявления), `size` (~36%), `floor_no` (~9%) и `building_age` (~7%). Числовые пропуски мы заполним медианой, а `furnished` удалим.

In [ ]:
print('Количество дубликатов с учетом столбца id:', df.duplicated().sum())
print('Количество дубликатов БЕЗ столбца id:', df.drop(columns=['id']).duplicated().sum())

**Важное замечание по дубликатам:**
Если проверять дубликаты вместе со столбцом `id`, получается 0, так как сайт Zingat присваивает каждому объявлению уникальный номер. Но если исключить `id`, то в датасете обнаруживается **11 559** полных дубликатов (одинаковые квартиры, адреса и цены). Их необходимо удалить, чтобы они не завышали вес повторяющихся объектов при обучении.

In [ ]:
print('Цены <= 0:', (df['price'] <= 0).sum())
print('Площадь <= 0:', (df['size'] <= 0).sum())
df['price_currency'].value_counts(dropna=False)

Есть 44 объявления с ценой 0 и 1 с ценой отрицательной — это явные ошибки ввода, такие строки нужно будет убрать. Отрицательных/нулевых значений площади нет. Валюта в основном TRY, но встречаются USD, EUR, GBP (и 715 пропусков) — цены в разных валютах напрямую сравнивать нельзя, нужно будет привести к одной.

In [ ]:
df['building_age'].value_counts()

Значения building_age выглядят адекватно: 0 — новостройка, остальные — диапазоны лет ('6-10 arası' и т.п.), проверила на нереалистичные значения (отрицательный возраст, дом старше 100 лет и т.п.) — таких не нашлось.

In [ ]:
print('Не удалось распознать формат room_count:', df['rooms_total'].isna().sum())
df.loc[df['rooms_total'].isna(), 'room_count'].value_counts()

Нашла ещё одну аномалию: у 2898 объявлений room_count равен просто + без единой цифры — явно битые данные, из текста никак не понять, сколько там комнат. На этапе предобработки такие строки, скорее всего, придётся удалить или заполнить самой частой категорией.

### 1.5 Промежуточные выводы

По итогам первого этапа можно сформулировать несколько гипотез, которые проверю на следующих этапах:

1. **Тип объявления (продажа/аренда) сильнее всего влияет на цену** — на боксплоте и в корреляции разница между продажей и арендой хорошо видна.
2. **Площадь (size) положительно влияет на цену** — подтверждается диаграммой рассеяния и корреляцией (+0.39 после очистки от выбросов).
3. **Город/район важен для цены** — медианная цена по топ-городам заметно различается.
4. **Тип жилья (sub_type) тоже влияет на цену** — виллы и особняки в среднем в несколько раз дороже квартир и сборных домов.
5. **Без очистки от выбросов и пропусков модель работать не будет** — в `price` и `size` огромные выбросы (вплоть до 2 млрд и почти 1 млн м²), а `furnished` придётся удалить полностью из-за 100% пропусков.

## Этап 2. Предварительная обработка данных

На первом этапе (EDA) я изучила сырые данные и нашла много проблем: сильный разброс цен, пропуски, опечатки и выбросы. Чтобы подготовить датасет к машинному обучению, на этом этапе я выполню:
- Очистку от дубликатов, приведение цен к одной валюте (TRY) и фильтрацию объявлений;
- Корректное заполнение пропусков в комнатах, этажах, возрасте здания и площади;
- Устранение аномалий и выбросов, логарифмирование целевой переменной для стабилизации дисперсии;
- Feature Engineering: создание новых признаков и удаление признаков с утечкой таргета (data leakage);
- Кодирование категориальных признаков (One-Hot и Out-of-Fold Target Encoding);
- Масштабирование признаков через `StandardScaler` и разбиение на обучающую и тестовую выборки (80/20).

### 2.1 Работа с пропусками и базовая очистка данных

Задача проекта — прогнозирование стоимости **покупки** недвижимости. В датасете Zingat есть объявления трех типов (`listing_type`): 1 — продажа, 2 — аренда, 3 — прочее. Стоимость аренды измеряется тысячами лир в месяц, а стоимость покупки — сотнями тысяч и миллионами. Если их смешать, модель не сможет нормально обучиться, поэтому я оставляю только продажу (`listing_type == 1`).

Также я перевожу редкие цены в валютах (USD, EUR, GBP) в турецкие лиры по курсу 2019 года, удаляю 11.5 тыс. полных дубликатов и убираю бесполезные столбцы: `id` (технический номер), `type` (константа "Konut") и `furnished` (в нем 100% пропусков).

In [ ]:
# Фильтрация только объявлений о продаже недвижимости
df_stage2 = df[df['listing_type'] == 1].copy()
print(f"Строк после фильтрации продажи (listing_type == 1): {len(df_stage2):,}")

# Конвертация валют в TRY
rates_to_try = {'TRY': 1.0, 'USD': 5.7, 'EUR': 6.4, 'GBP': 7.3}
df_stage2['price'] = df_stage2['price'] * df_stage2['price_currency'].map(rates_to_try).fillna(1.0)

# Удаление неинформативных колонок и дубликатов
cols_to_drop_stage2 = ['id', 'type', 'furnished', 'listing_type', 'price_currency', 'rooms_total']
df_stage2 = df_stage2.drop(columns=[c for c in cols_to_drop_stage2 if c in df_stage2.columns], errors='ignore')

initial_len = len(df_stage2)
df_stage2 = df_stage2.drop_duplicates()
print(f"Удалено полных дубликатов: {initial_len - len(df_stage2):,}")
print(f"Осталось уникальных записей о продаже: {len(df_stage2):,}")

**Мини-вывод:**
В выборке остались только объявления о продаже (278 644 строки). Цены приведены к единой валюте (TRY), дубликаты устранены, а неинформативные колонки удалены. Теперь можно безопасно переходить к заполнению пропусков.

### Преобразование признаков и заполнение пропусков

В числовых признаках есть пропуски и текстовые форматы:
1. `room_count`: формат вида "2+1" или "3+1" я перевожу в сумму комнат (3 или 4). Битые строки с одиночным плюсом без цифр заменяю на `NaN` и заполняю медианой.
2. `building_age`: текстовые интервалы (например, "6-10 arası") перевожу в середину диапазона (8 лет), пропуски заполняю медианой.
3. `floor_no` и `total_floor_count`: привожу к числам, учитывая цокольные этажи (-1, -2), первый этаж (1) и пентхаусы (равны общей этажности дома).
4. `size`: пропуски в площади заполняю медианой сгруппированной по подтипу жилья (`sub_type`), так как площадь виллы и обычной квартиры сильно отличается.
5. `heating_type`: пропущенные типы отопления помечаю категорией "Не указано".

In [ ]:
import re

# 1. Парсинг количества комнат
def parse_rooms_clean(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    m = re.match(r'^(\d+)\+(\d+)$', s)
    if m:
        return int(m.group(1)) + int(m.group(2))
    m = re.match(r'^\+(\d+)$', s)
    if m:
        return int(m.group(1))
    m = re.match(r'^(\d+)$', s)
    if m:
        return int(m.group(1))
    return np.nan

df_stage2['room_count'] = df_stage2['room_count'].apply(parse_rooms_clean)
df_stage2['room_count'] = df_stage2['room_count'].fillna(df_stage2['room_count'].median())

# 2. Парсинг возраста здания (берем среднюю точку диапазона)
age_dict = {
    '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5,
    '6-10 arası': 8, '11-15 arası': 13, '16-20 arası': 18,
    '21-25 arası': 23, '26-30 arası': 28, '31-35 arası': 33,
    '36-40 arası': 38, '40 ve üzeri': 45
}
df_stage2['building_age'] = df_stage2['building_age'].astype(str).map(age_dict)
df_stage2['building_age'] = df_stage2['building_age'].fillna(df_stage2['building_age'].median())

# 3. Парсинг общей этажности
def parse_total_floors_clean(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if s == '10-20 arası': return 15.0
    if s == '20 ve üzeri': return 25.0
    m = re.match(r'^(\d+)$', s)
    if m: return float(m.group(1))
    return np.nan

df_stage2['total_floor_count'] = df_stage2['total_floor_count'].apply(parse_total_floors_clean)
df_stage2['total_floor_count'] = df_stage2['total_floor_count'].fillna(df_stage2['total_floor_count'].median())

# 4. Парсинг этажа квартиры
floor_words = {
    'yüksek giriş': 1.0, 'müstakil': 1.0, 'bahçe katı': 0.0,
    'giriş katı': 1.0, 'zemin kat': 1.0, 'kot 1': -1.0,
    'kot 2': -2.0, 'kot 3': -3.0, 'kot 4': -4.0, 'bodrum kat': -1.0
}

def parse_floor_clean(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if s in floor_words:
        return floor_words[s]
    if 'en üst' in s or 'çatı' in s:
        return -999.0
    if 'ara kat' in s:
        return -888.0
    m = re.match(r'^-?(\d+(?:\.\d+)?)$', s)
    if m:
        return float(m.group(1))
    return np.nan

df_stage2['floor_no'] = df_stage2['floor_no'].apply(parse_floor_clean)
df_stage2.loc[df_stage2['floor_no'] == -999.0, 'floor_no'] = df_stage2.loc[df_stage2['floor_no'] == -999.0, 'total_floor_count']
df_stage2.loc[df_stage2['floor_no'] == -888.0, 'floor_no'] = (df_stage2.loc[df_stage2['floor_no'] == -888.0, 'total_floor_count'] / 2).round()
df_stage2['floor_no'] = df_stage2['floor_no'].fillna(df_stage2['floor_no'].median())

# 5. Заполнение пропусков в площади (групповая медиана по типу жилья) и отоплении
df_stage2['size'] = df_stage2['size'].fillna(df_stage2.groupby('sub_type')['size'].transform('median'))
df_stage2['size'] = df_stage2['size'].fillna(df_stage2['size'].median())
df_stage2['heating_type'] = df_stage2['heating_type'].fillna('Не указано')

print("Остаток пропущенных значений в основных столбцах:")
print(df_stage2[['room_count', 'building_age', 'total_floor_count', 'floor_no', 'size', 'heating_type']].isna().sum())

**Мини-вывод:**
Все строковые форматы комнат, этажей и возраста здания переведены в строгий числовой вид. Пропуски устранены с учетом логики данных (для площади использована групповая медиана по типам жилья). Во всех основных признаках пропусков теперь ровно 0.

### 2.2 Обработка выбросов, аномалий и логарифмирование цены

На этапе EDA были обнаружены сильные аномалии (квартиры по миллиону кв. метров, цены в 0 или миллиарды лир). Я отсекаю физически нереалистичные значения:
- Площадь от 15 до 1500 м²;
- Количество комнат от 1 до 10;
- Возраст здания от 0 до 60 лет;
- Этажность от 1 до 50 этажей.

Для цен я применяю обрезку по 1% и 99% квантилям, чтобы избавиться от экстремальных хвостов и опечаток. Кроме того, целевая переменная `price` имеет сильный правый скос, поэтому я применяю логарифмирование `log1p(price)`.

In [ ]:
# Фильтрация физических выбросов
n_before = len(df_stage2)
df_stage2 = df_stage2[
    (df_stage2['size'] >= 15) & (df_stage2['size'] <= 1500) &
    (df_stage2['room_count'] >= 1) & (df_stage2['room_count'] <= 10) &
    (df_stage2['building_age'] >= 0) & (df_stage2['building_age'] <= 60) &
    (df_stage2['total_floor_count'] >= 1) & (df_stage2['total_floor_count'] <= 50)
]

# Обрезка квантилей по цене (1% и 99%)
q_low = df_stage2['price'].quantile(0.01)
q_high = df_stage2['price'].quantile(0.99)
print(f"Границы цены (1% - 99%): от {q_low:,.0f} до {q_high:,.0f} TRY")
df_stage2 = df_stage2[(df_stage2['price'] >= q_low) & (df_stage2['price'] <= q_high)]

print(f"Отсечено выбросов и аномалий: {n_before - len(df_stage2):,} строк (всего {((n_before - len(df_stage2))/n_before)*100:.1f}%)")
print(f"Осталось чистых строк: {len(df_stage2):,}")

# Проверка эффекта логарифмирования цены
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

skew_raw = df_stage2['price'].skew()
axes[0].hist(df_stage2['price'] / 1000, bins=50, color='skyblue', edgecolor='black')
axes[0].set_title(f'Исходная цена (тыс. TRY)\nАсимметрия: {skew_raw:.2f}')
axes[0].set_xlabel('Цена (тыс. TRY)')
axes[0].set_ylabel('Количество')
axes[0].grid(True, alpha=0.3)

log_price = np.log1p(df_stage2['price'])
skew_log = log_price.skew()
axes[1].hist(log_price, bins=50, color='salmon', edgecolor='black')
axes[1].set_title(f'Логарифм цены log1p(price)\nАсимметрия: {skew_log:.2f}')
axes[1].set_xlabel('log1p(price)')
axes[1].set_ylabel('Количество')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Мини-вывод:**
Фильтрация отсекла всего 2.5% самых зашумленных данных, сохранив более 271 тыс. качественных наблюдений. Логарифмирование `log1p(price)` сработало отлично: коэффициент асимметрии упал с 308 до 0.26, и распределение цены стало близким к нормальному колоколу, что критично для обучения линейных и градиентных моделей.

### 2.3 Feature Engineering (Создание новых признаков)

Чтобы повысить качество будущих моделей, я создаю дополнительные признаки:
1. `city` и `district`: выделяю город и район из текстового поля адреса.
2. `size_per_room`: средняя площадь на одну комнату (`size / room_count`), отражает просторность объекта.
3. `floor_ratio`: относительный этаж (`floor_no / total_floor_count`), показывает положение квартиры по высоте здания.
4. `is_first_floor` и `is_last_floor`: бинарные флаги крайних этажей (первый этаж обычно дешевле, а последний может быть как пентхаусом, так и мансардой).
5. `start_month`: месяц публикации объявления (из `start_date`) для учета фактора сезонности.

**Устранение утечки данных (Data Leakage):**  
Столбец `tom` (время нахождения объявления на рынке) удаляю, так как на момент оценки нового объекта недвижимости в приложении этот параметр еще неизвестен. Также удаляю даты и исходную строку адреса.

In [ ]:
# 1. Выделение географии из адреса
addr_parts = df_stage2['address'].astype(str).str.split('/', expand=True)
df_stage2['city'] = addr_parts[0].str.strip().fillna('Другой')
df_stage2['district'] = addr_parts[1].str.strip().fillna('Другой') if addr_parts.shape[1] > 1 else 'Другой'

# 2. Новые расчетные признаки
df_stage2['size_per_room'] = df_stage2['size'] / df_stage2['room_count'].clip(lower=1)
df_stage2['floor_ratio'] = (df_stage2['floor_no'] / df_stage2['total_floor_count'].clip(lower=1)).clip(lower=0.0, upper=1.0)
df_stage2['is_first_floor'] = (df_stage2['floor_no'] <= 1).astype(int)
df_stage2['is_last_floor'] = (df_stage2['floor_no'] >= df_stage2['total_floor_count']).astype(int)
df_stage2['start_month'] = pd.to_datetime(df_stage2['start_date'], format='%m/%d/%y', errors='coerce').dt.month.fillna(1).astype(int)

# 3. Удаление столбцов с риском утечки данных и вспомогательных полей
leak_cols = ['start_date', 'end_date', 'tom', 'address']
df_stage2 = df_stage2.drop(columns=[c for c in leak_cols if c in df_stage2.columns], errors='ignore')

# 4. Проверка корреляций новых числовых признаков с ценой
eng_numeric = ['size', 'room_count', 'building_age', 'total_floor_count', 'floor_no', 
               'size_per_room', 'floor_ratio', 'is_first_floor', 'is_last_floor', 'start_month']

corr_with_price = df_stage2[eng_numeric].apply(lambda c: c.corr(np.log1p(df_stage2['price']))).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(x=corr_with_price.values, y=corr_with_price.index, palette='viridis')
plt.title('Корреляция признаков с логарифмом цены log1p(price)', fontsize=12)
plt.xlabel('Коэффициент корреляции Пирсона')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

display(pd.DataFrame({'Корреляция с log1p(price)': corr_with_price}))

**Мини-вывод:**
Сконструированные признаки хорошо согласуются с рыночной логикой: `size_per_room` показал корреляцию +0.44 с логарифмом цены, а признаки крайних этажей помогают уловить специфику ценообразования. Все признаки с риском утечки данных (`tom`, даты) удалены.

### 2.4 Кодирование категориальных признаков

Категории в датасете имеют разную кардинальность:
- `sub_type` (подтип жилья) и `heating_type` (отопление) имеют до 15 категорий. Их я кодирую с помощью **One-Hot Encoding** с параметром `drop_first=True` (чтобы не создавать линейную зависимость между дамми-столбцами).
- `city` (81 город) и `district` (~900 районов) имеют слишком много уникальных значений. Использовать One-Hot нельзя, иначе матрица раздуется на сотни столбцов и модель переобучится. Поэтому для них я применяю **Target Encoding** (кодирование средней логарифмической ценой по району/городу).

> Для предотвращения утечки данных (data leakage) на обучающей выборке используется схема **Out-of-Fold (K-Fold со сглаживанием)**, а для теста используются средние, посчитанные строго по train-выборке.

In [ ]:
from sklearn.model_selection import train_test_split, KFold

# Разделяем на признаки и таргет
X = df_stage2.drop(columns=['price']).copy()
y = np.log1p(df_stage2['price']).copy()

# Разделение на train и test (80% / 20%) с фиксацией random_state
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Размерность X_train: {X_train.shape}, размерность X_test: {X_test.shape}")

# Функция сглаженного Out-of-Fold Target Encoding
def target_encode_oof(train_df, test_df, col, y_tr, n_splits=5, smoothing=10, random_state=42):
    oof = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    global_mean = y_tr.mean()
    
    for tr_idx, val_idx in kf.split(train_df):
        tr_cat = train_df[col].iloc[tr_idx]
        val_cat = train_df[col].iloc[val_idx]
        tr_target = y_tr.iloc[tr_idx]
        
        means = tr_target.groupby(tr_cat).mean()
        counts = tr_target.groupby(tr_cat).count()
        smoothed = (means * counts + global_mean * smoothing) / (counts + smoothing)
        oof[val_idx] = val_cat.map(smoothed).fillna(global_mean).values
        
    # Для теста считаем сглаженные средние по всему train
    full_means = y_tr.groupby(train_df[col]).mean()
    full_counts = y_tr.groupby(train_df[col]).count()
    full_smoothed = (full_means * full_counts + global_mean * smoothing) / (full_counts + smoothing)
    test_enc = test_df[col].map(full_smoothed).fillna(global_mean).values
    
    return oof, test_enc

# Применяем Target Encoding для города и района
for geo_col in ['city', 'district']:
    oof_tr, test_enc = target_encode_oof(X_train, X_test, geo_col, y_train)
    X_train[f'{geo_col}_te'] = oof_tr
    X_test[f'{geo_col}_te'] = test_enc
    X_train = X_train.drop(columns=[geo_col])
    X_test = X_test.drop(columns=[geo_col])
    print(f"Колонка {geo_col} успешно закодирована в {geo_col}_te")

# One-Hot Encoding для подтипа жилья и отопления
ohe_cols = ['sub_type', 'heating_type']
X_train = pd.get_dummies(X_train, columns=ohe_cols, drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=ohe_cols, drop_first=True, dtype=float)

# Гарантируем одинаковый набор колонок между train и test
X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

print(f"\nРазмерность X_train после кодирования: {X_train.shape}")
print(f"Размерность X_test после кодирования:  {X_test.shape}")

**Мини-вывод:**
Категории закодированы без раздувания матрицы признаков и строго без утечки данных: типы жилья и отопления закодированы через One-Hot (39 колонок в итоге), а сложная география компактно выражена через Out-of-Fold Target Encoding.

### 2.5 Масштабирование признаков (Feature Scaling)

**Обоснование выбора метода:**
- `MinMaxScaler` сильно реагирует даже на единичные краевые значения.
- `RobustScaler` хорош при наличии нерешенных выбросов, но мы их уже убрали.
- `StandardScaler` приводит каждый признак к среднему 0 и дисперсии 1. Для линейных регрессий (Ridge, Lasso) это необходимо, чтобы веса признаков штрафовались равномерно и алгоритмы градиентного спуска сходились быстро. Деревьям решений масштаб не важен, но для линейных моделей стандартизация обязательна.

Я обучаю `StandardScaler` строго на обучающей выборке `X_train` (`fit_transform`), а к тестовой выборке `X_test` применяю только трансформацию (`transform`).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Инициализируем и обучаем StandardScaler строго на train
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Проверка среднего и стандартного отклонения первых 5 признаков после масштабирования:")
stats_check = pd.DataFrame({
    'Mean (train)': X_train_scaled.iloc[:, :5].mean().round(3),
    'Std (train)': X_train_scaled.iloc[:, :5].std().round(3),
    'Mean (test)': X_test_scaled.iloc[:, :5].mean().round(3),
    'Std (test)': X_test_scaled.iloc[:, :5].std().round(3)
})
display(stats_check)

**Мини-вывод:**
Все числовые признаки успешно стандартизированы. На обучающей выборке среднее равно 0.0, а стандартное отклонение 1.0. Скалер обучен строго на train без утечки информации о тестовой выборке.

### 2.6 Разбиение данных и сохранение результатов

Выборка разбита на train (80%) и test (20%) с фиксацией `random_state=42`. Проверяю, что в матрицах нет пропусков (NaN) и бесконечных значений (Inf). Очищенный датасет сохраняю в CSV, а готовые для моделей матрицы сохраняю в `preprocessed_data.pkl`.

In [ ]:
import joblib

# Финальная проверка на отсутствие пропусков и бесконечностей
assert X_train_scaled.isna().sum().sum() == 0, "Есть NaN в X_train_scaled!"
assert X_test_scaled.isna().sum().sum() == 0, "Есть NaN в X_test_scaled!"
assert not np.isinf(X_train_scaled.values).any(), "Есть Inf в X_train_scaled!"
assert not np.isinf(X_test_scaled.values).any(), "Есть Inf в X_test_scaled!"

print("=== РЕЗУЛЬТАТЫ ФИНАЛЬНОЙ ПРОВЕРКИ ===")
print(f"Обучающая выборка (train): {X_train_scaled.shape[0]:,} объектов, {X_train_scaled.shape[1]} признаков")
print(f"Тестовая выборка (test):    {X_test_scaled.shape[0]:,} объектов, {X_test_scaled.shape[1]} признаков")
print(f"Целевая переменная:        log1p(price), среднее = {y_train.mean():.2f}, медиана = {y_train.median():.2f}")

# Сохранение очищенного датасета в CSV
df_stage2.to_csv('cleaned_real_estate_stage1.csv', index=False)
print("Очищенный датасет успешно сохранен в cleaned_real_estate_stage1.csv")

# Сохранение словаря подготовленных данных для Этапа 3
preprocessed_dict = {
    'X_train': X_train,
    'X_test': X_test,
    'X_train_scaled': X_train_scaled,
    'X_test_scaled': X_test_scaled,
    'y_train': y_train,
    'y_test': y_test,
    'scaler': scaler,
    'features': X_train.columns.tolist()
}
joblib.dump(preprocessed_dict, 'preprocessed_data.pkl')
print("Подготовленные выборки успешно сохранены в preprocessed_data.pkl!")

### Промежуточные выводы

**Главные результаты предобработки:**

1. **Изменение размерности датасета:**
   - **Было:** 403 487 строк и 17 исходных признаков.
   - **Стало:** 271 764 строки и 39 признаков после Feature Engineering и кодирования (удалено 32.6% строк: аренда, 11.5 тыс. дубликатов и 2.5% выбросов).
   - **Итоговые выборки:** `train` — 217 411 строк (80%), `test` — 54 353 строки (20%).

2. **Качество данных и целевая переменная:**
   - Все пропуски устранены (**0 во всех колонках**), валюты приведены к TRY, удалены лишние поля (`id`, `type`, пустой `furnished`, `rooms_total`).
   - Логарифмирование `log1p(price)` снизило асимметрию цены с **308** до **0.26**, распределение стало колоколообразным.

3. **Признаки и масштабирование:**
   - Добавлено 5 новых признаков (самый сильный — `size_per_room` с корреляцией +0.44), удалены утечки данных (`tom`, даты).
   - Категории закодированы (One-Hot + Out-of-Fold Target Encoding), признаки отмасштабированы через `StandardScaler` строго на train.